# Project 1

By: Charles Ciampa and Abhishek Narang

In [1]:
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import convolve, convolve1d, correlate1d
import os

In [2]:
#setup function
def load_video_frames(frames_path):
    """Load image frames from a folder and convert them to a NumPy array.

    Args:
        frames_path (str or Path): Path to the folder containing .jpg frames.

    Returns:
        np.ndarray: Array of frames with shape (N, H, W, C) as float64.
    """
    #Load image frames from a folder and convert to grayscale.
    frames_dir = Path(frames_path)
    #Get all image files and sort them 
    image_files = sorted(frames_dir.glob('*.jpg'))
    # Sort numerically 
    try:
        image_files = sorted(image_files, key=lambda x: int(''.join(filter(str.isdigit, x.stem))))
    except:
        image_files = sorted(image_files)
    
    # Imports the image -> Converts type to float -> To numpy array
    frames = np.array([cv2.imread(path).astype(np.float64) for path in image_files])
    print(f"Loaded {len(frames)} from {frames_path}")
    return frames


In [3]:
def grayscale_image(img_arr:np.ndarray, rgb_weights = [1/3, 1/3, 1/3]) -> np.ndarray:
    """Converts the image to grayscale using the given weights.

    Args:
        img_arr (np.ndarray): An array of images with last dimension being RGB.
        rgb_weights (Optional | Array): The weights [R, G, B] to convert into an grayscale image. Defaults to average [1/3, 1/3, 1/3].

    Returns:
        Array (Numpy): The image where the final dimension no longer exists and is now a grayscale value.
    """
    # Grayscale the image (T, Height, Width, RGB) -> (T, Height, Width)
    return np.dot(img_arr, rgb_weights)

In [4]:
def create_gaussian_derivative_1d(sigma):
    
    #Create 1D Gaussian derivative kernel

    kernel_size = int(2 * np.ceil(3 * sigma) + 1)
    center = kernel_size // 2
    
    x = np.arange(kernel_size) - center
    #Derivative of Gaussian: -x/(sigma^2) * exp(-x^2/(2*sigma^2))
    kernel = -x / (sigma**2) * np.exp(-x**2 / (2 * sigma**2))
    kernel = kernel / np.sum(np.abs(kernel))  #normalize
    
    return kernel

In [5]:
#pick threshold
def estimate_noise_std(derivatives):
    """
    Estimate standard deviation of background noise from temporal derivatives.
    Most pixels are background with small temporal changes, model them as Gaussian noise with zero mean
    Returns: Estimated standard deviation of noise
    """
    #Using median absolute deviation for robust estimation
    mad = np.median(np.abs(derivatives))
    
    #for Gaussian distribution: std = 1.4826*MAD
    std_estimate = 1.4826 * mad
    
    return std_estimate

In [6]:
def select_threshold(derivatives, n_std=3.0):
    #Select threshold based on noise statistics.
 
    noise_std = estimate_noise_std(derivatives)
    threshold = n_std * noise_std
    return threshold


In [7]:
#motion detection process
# def detect_motion(frames, frame_idx, temporal_method='simple', t_sigma=1.0, 
#                   threshold_method='adaptive', manual_threshold=None, n_std=3.0):
#     """
#     Complete motion detection pipeline for a single frame.
#     Args: frames: List of video frames, frame_idx: Index of frame to process
#         temporal_method: 'simple' or 'gaussian', t_sigma: Temporal Gaussian std dev (if using gaussian)
#         threshold_method: 'adaptive' or 'manual', manual_threshold: Manual threshold value
#         n_std: Number of std devs for adaptive threshold
#     Returns: Dictionary with results
#     """
#     #1- Compute temporal derivative
#     if temporal_method == 'simple':
#         derivative = compute_temporal_derivative_simple(frames, frame_idx)
#     elif temporal_method == 'gaussian':
#         derivative = compute_temporal_derivative_gaussian(frames, frame_idx, t_sigma)
#     else:
#         raise ValueError(f"Unknown temporal method: {temporal_method}")
    
#     #2- Compute absolute value
#     abs_derivative = np.abs(derivative)
    
#     #3- Select threshold
#     if threshold_method == 'adaptive':
#         threshold = select_threshold(abs_derivative, n_std)
#     else:
#         threshold = manual_threshold
    
#     #4-Create binary mask
#     mask = (abs_derivative > threshold).astype(np.uint8)
    
#     #5- Overlay mask on original frame
#     result_frame = frames[frame_idx].copy()
#     overlay = np.stack([result_frame, result_frame, result_frame], axis=2)
#     overlay[:, :, 0] = np.maximum(overlay[:, :, 0], mask.astype(np.float32))
    
#     return {
#         'derivative': derivative,
#         'abs_derivative': abs_derivative,
#         'threshold': threshold,
#         'mask': mask,
#         'overlay': overlay,
#         'noise_std': estimate_noise_std(abs_derivative)
#     }


In [8]:
# #the 15pt part, temporal derivative filter
# def experiment_temporal_filters(frames, frame_idx):
    
#     #Compare simple and Gaussian temporal derivative filters - 3 values of sigma
    
    
#     #Test parameters-3 different t_sigma values
#     t_sigmas = [0.5, 1.0, 2.0]
#     fig, axes = plt.subplots(2, len(t_sigmas) + 1, figsize=(16, 8))
    
#     #Simple filter: 0.5[-1, 0, 1]
#     result_simple = detect_motion(frames, frame_idx, temporal_method='simple')
    
#     axes[0, 0].imshow(result_simple['abs_derivative'], cmap='hot')
#     axes[0, 0].set_title('Simple Filter\n[-1, 0, 1]')
#     axes[0, 0].axis('off')
    
#     axes[1, 0].imshow(result_simple['mask'], cmap='gray')
#     axes[1, 0].set_title(f"Mask (t={result_simple['threshold']:.4f})")
#     axes[1, 0].axis('off')
#     print(f"\n\n--------")

#     print(f"Temporal derivative filter- frame: "+ str(frame_idx))
#     print(f"\nSimple Filter:")
#     print(f"   Threshold: {result_simple['threshold']:.4f}")
#     print(f"   Noise std: {result_simple['noise_std']:.4f}")
#     print(f"   Motion pixels: {np.sum(result_simple['mask'])} ({100*np.mean(result_simple['mask']):.2f}%)")
    
#     #Gaussian filters with diff sigma values
#     for i, t_sigma in enumerate(t_sigmas): #1.0 and 2.0
#         result = detect_motion(frames, frame_idx, 
#                               temporal_method='gaussian', t_sigma=t_sigma)
        
#         axes[0, i+1].imshow(result['abs_derivative'], cmap='hot')
#         axes[0, i+1].set_title(f'Gaussian Derivative\nσ_t={t_sigma}')
#         axes[0, i+1].axis('off')
        
#         axes[1, i+1].imshow(result['mask'], cmap='gray')
#         axes[1, i+1].set_title(f"Mask (t={result['threshold']:.4f})")
#         axes[1, i+1].axis('off')
        
#         print(f"Gaussian Filter (σ_t={t_sigma}):")
#         print(f"  Threshold: {result['threshold']:.4f}")
#         print(f"  Noise std: {result['noise_std']:.4f}")
#         print(f"  Motion pixels: {np.sum(result['mask'])} ({100*np.mean(result['mask']):.2f}%)")
    
#     plt.tight_layout()
#     plt.savefig('temporal_filters_comparison.png', dpi=150, bbox_inches='tight')
#     plt.show()

In [9]:
def apply_temporal_kernel(img_arr, kernel, axis=0, **args):
    """Applies a 1D kernel along the temporal axis of the image array.

    Args:
        img_arr (np.ndarray): Numpy array (T, H, W) representing video frames.
        kernel (array-like): 1D kernel to convolve along the specified axis.
        axis (int, optional): Axis along which to apply the kernel. Defaults to 0 (temporal).
        **args: Additional keyword arguments passed to scipy.ndimage.convolve1d.

    Returns:
        np.ndarray: The filtered array with the same shape as img_arr.
    """
    # I switched to correlate because convolve will flip the kernel, which is not what we want for this case.
    return correlate1d(img_arr, kernel, axis=axis, **args)

In [10]:
def apply_img_kernel(img_arr, kernel, axes=(1,2), **args):
    """Applys a kernel to the image.

    Args:
        img_arr (numpy): Numpy array (T, H, W)
        kernel (numpy): (NXN) Kernel which to apply over the images

    Returns:
        numpy: The image post filter
    """
    # Convolve doesn't matter as the filter is symetrical, so the math works out to be the same
    return convolve(img_arr, kernel, axes=axes, **args)

In [11]:
def apply_temporal_and_image_kernel(rgb_img_arr, image_kernel, temporal_kernel, threshold: None | float = None, auto_thres_std: float = 3.0):
    # Convert to grayscale
    bw_imgs = grayscale_image(np.copy(rgb_img_arr))
    # Apply image kernel
    bw_imgs = apply_img_kernel(bw_imgs, image_kernel)
    # Scall Images from [0, 255] -> [0, 1]
    bw_imgs = bw_imgs / 255
    # Apply Temporal Kernel
    bw_imgs = apply_temporal_kernel(bw_imgs, temporal_kernel)
    thresh = select_threshold(bw_imgs, n_std=auto_thres_std) if threshold is None else threshold
    # Get boolean mask to apply over the images
    mask = np.abs(bw_imgs) > thresh
    # Apply the mask to the rgb img
    res_imgs = rgb_img_arr * mask[..., np.newaxis]
    # Return result
    return res_imgs


In [12]:
def export_to_mp4(img_arr: np.ndarray, path: str, overwrite_path: bool = True):
    if os.path.exists(path):
        if not overwrite_path:
            raise NameError("Path provided ({path}) already exists and overwrite set to false.")
        os.remove(path)

    _, H, W, _ = img_arr.shape

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(path, fourcc, 30, (W, H))

    for frame in img_arr.astype(np.uint8):
        out.write(frame)
    out.release()

    print("Export Completed")

In [13]:
def compute_kernel_combinations(img_arr, img_kernels, t_kernels):
    """Compute all combinations of image and temporal kernels for motion detection.

    Args:
        img_arr (np.ndarray): RGB image array with shape (T, H, W, C).
        img_kernels (list): List of tuples (name, kernel) for spatial filtering.
        t_kernels (list): List of tuples (name, kernel) for temporal filtering.

    Returns:
        list: List of tuples (name, result_array) for each kernel combination.
    """
    # All Kernal Combinations
    res = []
    # Itereate through all combinations of kernels
    for img_kernel_name, img_kernel in img_kernels:
        for t_name, t_kernel in t_kernels:
            # Computer combination and append to the resultant array
            res.append((f"{img_kernel_name} & {t_name}", apply_temporal_and_image_kernel(img_arr, img_kernel, t_kernel)))
    return res

Kernels to test on images

In [14]:
#Kernels
# Image Kernels
box_filter_3x3 = np.ones((3,3)) / 9
box_filter_5x5 = np.ones((5,5)) / 25

kernel_3x3_1d = cv2.getGaussianKernel(3, 0)  # size=3, sigma=0 (auto-calculated)
gaussian_3x3 =kernel_3x3_1d * kernel_3x3_1d.T

kernel_5x5_1d = cv2.getGaussianKernel(5, 0)

# All image kernels
all_img_kernels = [
    ('Box Filter 3x3', box_filter_3x3),
    ('Box Filter 5x5', box_filter_5x5),
    ('Gaussian 0.6 3x3', cv2.getGaussianKernel(3, 0.6) * cv2.getGaussianKernel(3, 0.6).T),
    ('Gaussian 1.0 5x5', cv2.getGaussianKernel(5, 1.0) * cv2.getGaussianKernel(5, 1.0).T),
    ('Gaussian 1.4 7x7', cv2.getGaussianKernel(7, 1.4) * cv2.getGaussianKernel(7, 1.4).T),
    ('Gaussian 1.8 9x9', cv2.getGaussianKernel(9, 1.8) * cv2.getGaussianKernel(9, 1.8).T),
    ('Gaussian 2.2 11x11', cv2.getGaussianKernel(11, 2.2) * cv2.getGaussianKernel(11, 2.2).T),
]

# Temporal Kernel
simple_temporal_kernel = np.array([-1, 0, 1]) / 2
gaussian_0_termporal_kernel = create_gaussian_derivative_1d(0.5)
gaussian_1_termporal_kernel = create_gaussian_derivative_1d(1.0)
gaussian_2_termporal_kernel = create_gaussian_derivative_1d(2.0)

# All Temporal Kernels
all_t_kernels = [
    ('Simple Temporal Kernel', simple_temporal_kernel),
    ('Gaussian Temporal Kernel (σ=0.5)', gaussian_0_termporal_kernel),
    ('Gaussian Temporal Kernel (σ=1.0)', gaussian_1_termporal_kernel),
    ('Gaussian Temporal Kernel (σ=2.0)', gaussian_2_termporal_kernel),
]

## Computing Results

### Red Chair Images

In [15]:
# Import the images
red_chair_imgs_rgb = load_video_frames("data/RedChair")
print(red_chair_imgs_rgb.shape)

Loaded 353 from data/RedChair
(353, 240, 320, 3)


In [16]:
all_red_chair_combinations = compute_kernel_combinations(red_chair_imgs_rgb, all_img_kernels, all_t_kernels)
print(all_red_chair_combinations[0][1].shape)

(353, 240, 320, 3)


In [17]:
# Save all the results
for name, movie in all_red_chair_combinations:
    export_to_mp4(movie, f"results/RedChair/{name}.mp4")

Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed


### Office

In [18]:
# Import the images
office_imgs_rgb = load_video_frames("data/Office")

Loaded 1070 from data/Office


In [19]:
office_combinations = compute_kernel_combinations(office_imgs_rgb, all_img_kernels, all_t_kernels)
print(office_combinations[0][1].shape)

(1070, 240, 320, 3)


In [20]:
# Save all the results
for name, movie in office_combinations:
    export_to_mp4(movie, f"results/Office/{name}.mp4")

Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed


### Enter Exit Crossing Paths 2 Cor

In [21]:
# Import the images
exit_imgs_rgb = load_video_frames("data/EnterExitCrossingPaths2cor")
print(exit_imgs_rgb.shape)

Loaded 485 from data/EnterExitCrossingPaths2cor
(485, 288, 384, 3)


In [22]:
exit_combinations = compute_kernel_combinations(exit_imgs_rgb, all_img_kernels, all_t_kernels)
print(exit_combinations[0][1].shape)

(485, 288, 384, 3)


In [23]:
# Save all the results
for name, movie in exit_combinations:
    export_to_mp4(movie, f"results/EnterExitCrossingPaths2cor/{name}.mp4")

Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed
Export Completed


In [25]:
print(all_img_kernels[2])

('Gaussian 0.6 3x3', array([[0.02768181, 0.11101489, 0.02768181],
       [0.11101489, 0.44521319, 0.11101489],
       [0.02768181, 0.11101489, 0.02768181]]))
